# 11 - Cost-sensitive occupancy and score calibration (YOLO11n and YOLO11s)
A **false vacant** (an occupied slot reported free) sends a driver to a full slot; a **false occupied** hides a free one. Spec section 14: with a *calibrated* occupancy probability P(O), report occupied when **P(O) > C_FO / (C_FO + C_FV)**. The slot score from notebook 07 (max confidence of the supporting detections) is **not** a probability, so this notebook:

1. **Calibrates** the score on val: Platt (logistic) and isotonic (step function), with 5-fold cross-fitting grouped by camera-day, so val calibration quality is measured on data the calibrator did not see. Brier score, log loss, ECE and reliability diagrams; test reported once with calibrators fitted on all of val.
2. **Cost ratios** C_FV / C_FO in {0.5, 1, 2, 5, 10, 20} (C_FO = 1). Three decision rules per ratio:
   - *notebook 07 score threshold* (cost-agnostic, max val macro-F1),
   - *val score threshold*: the score threshold minimising the val cost-weighted error (a **score** threshold, not a probability),
   - *calibrated + Bayes threshold*: P(O) > 1 / (1 + ratio), no threshold tuning.
   The rule used for each ratio is chosen by its **cross-fitted val cost** (each fold's threshold / calibrator comes from the other folds); test is reported for every rule.
3. **Temporal filter in cost terms**: notebook 08 chose a gated filter (k=2, gate 0.75) that halves false flips but raises false vacants, because it delays arrivals. An **asymmetric** filter (flip to occupied at once, to vacant only after k_vac frames) is compared in cost per ratio; choice on val.

Metrics: false-vacant rate (FV / occupied), false-occupied rate (FO / vacant), cost-weighted error per 1000 slot decisions, accuracy, macro-F1, coverage (share of slots with a decision; slots outside the annotated area are *unknown*). CPU only.

**Inputs (Add Input -> Your Work):** notebook 07 output (`artifacts/occupancy/`) and notebook 03 output (`artifacts/splits/`).
**Accelerator:** none. Internet ON only to clone the repo.
**How to run:** `SMOKE = True` + Run All (a subset of camera-days). Then `SMOKE = False` + **Save Version -> Save & Run All**.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Settings and inputs
SMOKE = True                      # True: a subset of camera-days. False: everything
SMOKE_GROUPS = 12
VARIANTS = ["yolo11n", "yolo11s"]
RATIOS = [0.5, 1, 2, 5, 10, 20]   # C_FV / C_FO, with C_FO = 1
FOLDS = 5                         # val cross-fitting, grouped by camera + day
BINS = 10                         # reliability / ECE bins
TOL_FRAMES = 3; MAX_GAP_FACTOR = 3.0   # same as notebook 08
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

import os, json, time, csv, gzip
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import numpy as np
from src.common.config import load_yaml, config_hash
from src.common.status import new_run_id, write_status
from src.training import yolo_train as yt
from src.evaluation import det_eval as de
from src.occupancy import association as asc
from src.occupancy import cost_sensitive as cs
from src.temporal import filters as tf
OCFG = load_yaml("configs/occupancy.yaml"); TCFG = load_yaml("configs/temporal.yaml")

mans = [p for p in yt.find_files(INPUT_ROOT, "split_manifest.csv") if p.parent.name == "splits"]
assert mans, "split_manifest.csv not found: attach notebook 03's output"
ROW_BY_ID = {int(r["image_id"]): r for r in de.read_manifest(mans[0])}
SPLIT_INFO = json.loads((mans[0].parent / "split_info.json").read_text())
occ = [p.parent for p in yt.find_files(INPUT_ROOT, "slot_scores_test.csv.gz") if p.parent.name == "occupancy"]
assert occ, "slot_scores_test.csv.gz not found: attach notebook 07's output (full run)"
OCC = occ[0]; st07 = json.loads((OCC / "status.json").read_text())
assert st07["state"] == "COMPLETE"
THR07 = {}
for v in VARIANTS:
    rec = json.loads((OCC / "occupancy_record_{}.json".format(v)).read_text())
    assert not rec["smoke_test"] and rec["dataset_split_hash"] == SPLIT_INFO["manifest_hash"], v
    THR07[v] = float(rec["chosen_on_val"]["conf_thr"])
    assert abs(OCFG["decision"]["score_threshold"][v] - THR07[v]) < 1e-9, "configs/occupancy.yaml threshold differs from notebook 07"
GATED08 = {v: dict(TCFG["per_model"][v]) for v in VARIANTS}
print("notebook 07:", OCC, "| run", st07["run_id"], "| score thresholds", THR07, "| notebook 08 filter", GATED08)

def parse_ts(s):
    return datetime.fromisoformat(s.replace("Z", "")).timestamp()

def load_scores(split):
    cols = defaultdict(list)
    with gzip.open(str(OCC / "slot_scores_{}.csv.gz".format(split)), "rt") as f:
        for r in csv.DictReader(f):
            for k in ("image", "camera", "timestamp", "slot_id"):
                cols[k].append(r[k])
            cols["image_id"].append(int(r["image_id"])); cols["gt"].append(int(r["gt_occupied"])); cols["valid"].append(int(r["valid"]))
            for v in VARIANTS:
                cols[v].append(float(r["score_" + v]))
    o = {k: np.array(x) for k, x in cols.items()}
    o["day"] = np.array([ROW_BY_ID[int(i)]["day"] for i in o["image_id"]])
    o["site"] = np.array([ROW_BY_ID[int(i)]["site"] for i in o["image_id"]])
    o["group"] = np.char.add(np.char.add(o["camera"].astype(str), "|"), o["day"].astype(str))
    if SMOKE:
        u = np.unique(o["group"]); keep = u[np.linspace(0, len(u) - 1, min(SMOKE_GROUPS, len(u))).astype(int)]
        m = np.isin(o["group"], keep); o = {k: x[m] for k, x in o.items()}
    return o

t0 = time.time()
RAW = {s: load_scores(s) for s in ("val", "test")}
for s, o in RAW.items():
    vm = o["valid"].astype(bool)
    print("{:<5} slot rows {} | with a decision (coverage) {:.4f} | unknown (outside annotated area) {} | occupied share {:.3f} | camera-days {}".format(
        s, len(vm), vm.mean(), int((~vm).sum()), o["gt"][vm].mean(), len(np.unique(o["group"]))))
print("loaded in {:.0f} s".format(time.time() - t0))
OUT = Path(WORK) / "artifacts" / "cost_sensitive"
OUT.mkdir(parents=True, exist_ok=True)
SUF = "_smoke" if SMOKE else ""
CFG_HASH = config_hash({"occupancy": OCFG, "temporal": TCFG, "smoke": SMOKE, "smoke_groups": SMOKE_GROUPS, "ratios": RATIOS, "folds": FOLDS,
                        "bins": BINS, "split": SPLIT_INFO["manifest_hash"], "nb07": st07["run_id"]})
RUN_ID = new_run_id("cst")
STATUS = OUT / "status.json"
write_status(STATUS, "11_cost_sensitive_occupancy_and_calibration", RUN_ID, "RUNNING", CFG_HASH, "inputs loaded")

In [ ]:
# 1. Calibration on VAL (grouped cross-fitting) and TEST (fitted on all of val)
def valid(o, v):
    m = o["valid"].astype(bool)
    return o[v][m], o["gt"][m].astype(int), o["group"][m], o["site"][m]

def quality(p, y):
    return {"brier": round(cs.brier(p, y), 5), "log_loss": round(cs.log_loss(p, y), 5), "ece": round(cs.ece(p, y, BINS), 5)}

CAL, CALQ, OOF, PTEST, REL = {}, {}, {}, {}, {}
try:
    for v in VARIANTS:
        sv, yv, gv, _ = valid(RAW["val"], v); st_, yt_, _, site_t = valid(RAW["test"], v)
        folds = cs.group_folds(gv, FOLDS)
        CALQ[v] = {"raw score (not a probability)": {"val_cv": quality(np.clip(sv, 0, 1), yv), "test": quality(np.clip(st_, 0, 1), yt_)}}
        REL[v] = {"raw score": (cs.reliability(np.clip(sv, 0, 1), yv, BINS), cs.reliability(np.clip(st_, 0, 1), yt_, BINS))}
        OOF[v], PTEST[v], CAL[v] = {}, {}, {}
        for meth in cs.CALIBRATORS:
            OOF[v][meth] = cs.cross_fit(meth, sv, yv, folds)
            CAL[v][meth] = cs.make(meth).fit(sv, yv)
            PTEST[v][meth] = CAL[v][meth].predict(st_)
            CALQ[v][meth] = {"val_cv": quality(OOF[v][meth], yv), "test": quality(PTEST[v][meth], yt_),
                             "test_ece_per_site": {str(s): round(cs.ece(PTEST[v][meth][site_t == s], yt_[site_t == s], BINS), 5) for s in sorted(set(site_t))}}
            REL[v][meth] = (cs.reliability(OOF[v][meth], yv, BINS), cs.reliability(PTEST[v][meth], yt_, BINS))
        CALQ[v]["chosen"] = min(cs.CALIBRATORS, key=lambda m: (CALQ[v][m]["val_cv"]["log_loss"], CALQ[v][m]["val_cv"]["brier"]))
except Exception as e:
    write_status(STATUS, "11_cost_sensitive_occupancy_and_calibration", RUN_ID, "FAILED", CFG_HASH, repr(e)[:500]); raise
for v in VARIANTS:
    print("\n{} calibration (val: out-of-fold over {} camera-day groups; test: calibrator fitted on all of val). Chosen by val log loss: {}".format(
        v, len(np.unique(RAW["val"]["group"])), CALQ[v]["chosen"]))
    print("  {:<30} {:>10} {:>10} {:>9}   {:>10} {:>10} {:>9}".format("", "val brier", "val logL", "val ECE", "test brier", "test logL", "test ECE"))
    for meth in ["raw score (not a probability)"] + list(cs.CALIBRATORS):
        a, b = CALQ[v][meth]["val_cv"], CALQ[v][meth]["test"]
        print("  {:<30} {:>10} {:>10} {:>9}   {:>10} {:>10} {:>9}".format(meth, a["brier"], a["log_loss"], a["ece"], b["brier"], b["log_loss"], b["ece"]))
    print("  test ECE per site ({}): {}".format(CALQ[v]["chosen"], CALQ[v][CALQ[v]["chosen"]]["test_ece_per_site"]))
    if "platt" in CAL[v]:
        print("  Platt: P = 1 / (1 + exp(-({:.3f} * score + {:.3f})))".format(CAL[v]["platt"].a, CAL[v]["platt"].b))

In [ ]:
# 2. Cost-sensitive decisions for each cost ratio C_FV / C_FO
def equiv_score_thr(meth, cal, t):
    """Score from which the calibrated rule P(O) > t says occupied (for reading the probability rule as a score)."""
    if meth == "platt":
        return None if cal.a <= 0 else round((np.log(t / (1 - t)) - cal.b) / cal.a, 4)
    above = np.nonzero(cal.y > t)[0]
    return round(float(cal.x[above[0]]), 4) if len(above) else None

def row(v, split, ratio, rule, pred, y, extra):
    c = asc.confusion(y, pred)
    return {"model": v, "split": split, "ratio_fv_fo": ratio, "rule": rule, **extra, "FV": c["FV"], "FO": c["FO"],
            "false_vacant_rate": c["false_vacant_rate"], "false_occupied_rate": c["false_occupied_rate"],
            "cost_per_1000": round(cs.cost_per_1000(c["FV"], c["FO"], c["slots"], ratio, 1.0), 3),
            "accuracy": c["accuracy"], "macro_f1": c["macro_f1"], "decisions": c["slots"]}

ROWS, CHOICE = [], {}
for v in VARIANTS:
    sv, yv, gv, _ = valid(RAW["val"], v); st_, yt_, _, _ = valid(RAW["test"], v)
    folds = cs.group_folds(gv, FOLDS)
    cov = {s: round(float(RAW[s]["valid"].astype(bool).mean()), 4) for s in RAW}
    CHOICE[v] = {}
    for ratio in RATIOS:
        t_bayes = cs.bayes_threshold(ratio, 1.0)
        # cross-fitted val cost of each rule (the 07 threshold is fixed; the score threshold is re-chosen per fold)
        cv_pred = {"nb07 score threshold": sv >= THR07[v], "val score threshold": np.zeros(len(sv), bool)}
        for f in np.unique(folds):
            te = folds == f
            b = cs.best_score_threshold(sv[~te], yv[~te], ratio, 1.0, prefer=THR07[v])
            cv_pred["val score threshold"][te] = sv[te] >= b["thr"]
        for meth in cs.CALIBRATORS:
            cv_pred["calibrated {} + Bayes".format(meth)] = OOF[v][meth] > t_bayes
        cv_cost = {k: round(cs.cost_per_1000(int((yv.astype(bool) & ~p).sum()), int((~yv.astype(bool) & p).sum()), len(yv), ratio, 1.0), 3)
                   for k, p in cv_pred.items()}
        best = cs.best_score_threshold(sv, yv, ratio, 1.0, prefer=THR07[v])
        rules = {"nb07 score threshold": (st_ >= THR07[v], sv >= THR07[v], {"threshold": THR07[v], "threshold_type": "score"}),
                 "val score threshold": (st_ >= best["thr"], sv >= best["thr"], {"threshold": best["thr"], "threshold_type": "score"})}
        for meth in cs.CALIBRATORS:
            rules["calibrated {} + Bayes".format(meth)] = (PTEST[v][meth] > t_bayes, CAL[v][meth].predict(sv) > t_bayes,
                {"threshold": round(t_bayes, 4), "threshold_type": "probability ({})".format(meth),
                 "equivalent_score_threshold": equiv_score_thr(meth, CAL[v][meth], t_bayes)})
        chosen = min(cv_cost, key=lambda k: (cv_cost[k], list(cv_cost).index(k)))
        CHOICE[v][ratio] = {"rule": chosen, "val_cv_cost_per_1000": cv_cost, **rules[chosen][2]}
        for name, (pt, pv, ex) in rules.items():
            ex = dict(ex, val_cv_cost_per_1000=cv_cost[name], chosen_on_val=name == chosen, coverage=cov["test"])
            ROWS.append(row(v, "test", ratio, name, pt, yt_, ex))
            ROWS.append(row(v, "val_in_sample", ratio, name, pv, yv, dict(ex, coverage=cov["val"])))
with open(str(OUT / "cost_rules{}.csv".format(SUF)), "w", newline="") as f:
    keys = list(dict.fromkeys(k for r in ROWS for k in r))
    w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(ROWS)

H = "  {:>5} {:<28} {:>9} {:>9} {:>6} {:>6} {:>8} {:>8} {:>9} {:>9} {:>8}"
print("TEST, every rule at every cost ratio (cost per 1000 slot decisions with C_FO = 1; '*' = rule chosen on val by cross-fitted cost)")
for v in VARIANTS:
    print("\n" + v); print(H.format("ratio", "rule", "thr", "score eq", "FV", "FO", "FV rate", "FO rate", "cost/1k", "val cv", "macroF1"))
    for r in ROWS:
        if r["model"] == v and r["split"] == "test":
            print(H.format(r["ratio_fv_fo"], ("* " if r["chosen_on_val"] else "  ") + r["rule"], r["threshold"], str(r.get("equivalent_score_threshold", "")),
                           r["FV"], r["FO"], r["false_vacant_rate"], r["false_occupied_rate"], r["cost_per_1000"], r["val_cv_cost_per_1000"], r["macro_f1"]))

In [ ]:
# 3. Temporal filter in cost terms: none vs notebook 08 gated vs asymmetric (flip to vacant only after k_vac frames)
def build_sequences(o):
    names, first = np.unique(o["image"], return_index=True)
    fr = np.argsort(first); names, first = names[fr], first[fr]
    idx_of_name = {n: i for i, n in enumerate(names)}
    f_of_row = np.array([idx_of_name[n] for n in o["image"]])
    cam = o["camera"][first]; sec = np.array([parse_ts(t) for t in o["timestamp"][first]]); day = o["day"][first]
    sp = tf.split_sequences(cam, sec, day, MAX_GAP_FACTOR)
    slots_of_cam = {c: sorted(set(o["slot_id"][o["camera"] == c])) for c in np.unique(cam)}
    seqs = []
    for q in range(sp["seq"].max() + 1 if len(sp["seq"]) else 0):
        frames = np.nonzero(sp["seq"] == q)[0]; frames = frames[np.argsort(sec[frames], kind="stable")]
        c = cam[frames[0]]; T, S = len(frames), len(slots_of_cam[c])
        seqs.append({"frames": frames, "sec": sec[frames], "spacing_s": sp["spacing_s"][c], "col": {s: j for j, s in enumerate(slots_of_cam[c])},
                     "gt": np.full((T, S), -1, np.int8), "obs": np.zeros((T, S), bool), "score": {v: np.zeros((T, S)) for v in VARIANTS}})
    pos = np.zeros(len(names), int)
    for s in seqs:
        pos[s["frames"]] = np.arange(len(s["frames"]))
    for r in range(len(o["gt"])):
        f = f_of_row[r]; s = seqs[sp["seq"][f]]; t = pos[f]; j = s["col"][o["slot_id"][r]]
        if o["valid"][r]:
            s["obs"][t, j] = True; s["gt"][t, j] = o["gt"][r]
        for v in VARIANTS:
            s["score"][v][t, j] = o[v][r]
    return seqs

def tgrid(v):
    g = [{"method": "none"}, {k: x for k, x in GATED08[v].items() if k in ("method",) + tf.PARAMS}]
    return g + [{"method": "asymmetric", "k_occ": 1, "k_vac": kv, **({"gate": g} if g is not None else {})} for kv in (2, 3) for g in (None, 0.5, 0.75)]
TGRID = {v: tgrid(v) for v in VARIANTS}
def t_eval(v, seqs, cfg):
    gts, preds, parts, sf = [], [], [], 0
    for s in seqs:
        st = tf.run_sequence(s["score"][v], s["obs"], cfg, THR07[v]); m = s["obs"]
        gts.append(s["gt"][m]); preds.append(st[m]); sf += int(m.sum())
        parts.append(tf.transition_metrics(s["gt"], np.where(m, st, -1), s["sec"], TOL_FRAMES))
    c = asc.confusion(np.concatenate(gts), np.concatenate(preds)); t = tf.merge_transition_metrics(parts, sf, 1.0)
    return {"setting": tf.label(cfg), "cfg": cfg, "FV": c["FV"], "FO": c["FO"], "n": c["slots"], "macro_f1": c["macro_f1"],
            "false_per_1000": t["false_per_1000_slot_frames"], "missed": t["missed"], "delay_fr": t["mean_delay_frames"]}

t0 = time.time()
TSEQ = {s: build_sequences(RAW[s]) for s in RAW}
TEMP = {v: {s: [t_eval(v, TSEQ[s], cfg) for cfg in TGRID[v]] for s in TSEQ} for v in VARIANTS}
print("temporal settings evaluated in {:.0f} s (at the notebook 07 score thresholds)".format(time.time() - t0))
TCHOICE = {}
for v in VARIANTS:
    print("\n{}  (cost per 1000 slot-frames at each ratio; '*' = lowest val cost)".format(v))
    print("  {:<34} {:>5} {:>5} {:>9} {:>8} {:>8}  ".format("setting (TEST)", "FV", "FO", "false/1k", "delay_fr", "macroF1") + " ".join("r={:<5}".format(r) for r in RATIOS))
    TCHOICE[v] = {}
    for ratio in RATIOS:
        cost = lambda r: cs.cost_per_1000(r["FV"], r["FO"], r["n"], ratio, 1.0)
        i = min(range(len(TGRID[v])), key=lambda i: (round(cost(TEMP[v]["val"][i]), 6), TEMP[v]["val"][i]["false_per_1000"], i))
        TCHOICE[v][ratio] = {"setting": TGRID[v][i], "val_cost_per_1000": round(cost(TEMP[v]["val"][i]), 3), "test_cost_per_1000": round(cost(TEMP[v]["test"][i]), 3),
                             "test_cost_none": round(cost(TEMP[v]["test"][0]), 3), "test_cost_nb08": round(cost(TEMP[v]["test"][1]), 3)}
    for i, r in enumerate(TEMP[v]["test"]):
        cells_ = []
        for ratio in RATIOS:
            mark = "*" if TCHOICE[v][ratio]["setting"] is TGRID[v][i] else " "
            cells_.append("{:<7}".format("{:.2f}{}".format(cs.cost_per_1000(r["FV"], r["FO"], r["n"], ratio, 1.0), mark)))
        print("  {:<34} {:>5} {:>5} {:>9} {:>8} {:>8}  ".format(r["setting"], r["FV"], r["FO"], r["false_per_1000"], str(r["delay_fr"]), r["macro_f1"]) + " ".join(cells_))
with open(str(OUT / "temporal_cost{}.csv".format(SUF)), "w", newline="") as f:
    w = csv.writer(f); w.writerow(["model", "split", "setting", "FV", "FO", "n", "macro_f1", "false_per_1000", "missed", "delay_fr"])
    for v in VARIANTS:
        for s in TEMP[v]:
            for r in TEMP[v][s]:
                w.writerow([v, s, r["setting"], r["FV"], r["FO"], r["n"], r["macro_f1"], r["false_per_1000"], r["missed"], r["delay_fr"]])

In [ ]:
# Plots: reliability, cost vs ratio per rule, FV / FO rates vs ratio, thresholds vs ratio
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2 * len(VARIANTS), figsize=(5.2 * 2 * len(VARIANTS), 4.4), squeeze=False)
for k, v in enumerate(VARIANTS):
    for j, (split, lab) in enumerate(((0, "VAL out-of-fold"), (1, "TEST"))):
        a = ax[0][2 * k + j]; a.plot([0, 1], [0, 1], ls=":", color="0.5")
        for name, rel in REL[v].items():
            r_ = rel[split]; a.plot([x["mean_p"] for x in r_], [x["frac_occupied"] for x in r_], marker="o", ms=3, label=name)
        a.set(xlabel="predicted P(occupied) (raw: the score)", ylabel="observed occupied share", title="{} reliability, {}".format(v, lab)); a.grid(alpha=0.3); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(str(OUT / "reliability{}.png".format(SUF)), dpi=110); plt.show()

RULES = sorted({r["rule"] for r in ROWS}, key=lambda x: [r["rule"] for r in ROWS].index(x))
fig, ax = plt.subplots(len(VARIANTS), 3, figsize=(17, 4.2 * len(VARIANTS)), squeeze=False)
for k, v in enumerate(VARIANTS):
    T = [r for r in ROWS if r["model"] == v and r["split"] == "test"]
    for ci, rule in enumerate(RULES):
        rr = [r for r in T if r["rule"] == rule]; col = "C{}".format(ci)
        ax[k][0].plot(RATIOS, [r["cost_per_1000"] for r in rr], marker="o", color=col, label=rule)
        ax[k][1].plot(RATIOS, [r["false_vacant_rate"] for r in rr], marker="o", color=col, label=rule + " FV rate (solid)")
        ax[k][1].plot(RATIOS, [r["false_occupied_rate"] for r in rr], marker="x", ls="--", color=col, label=rule + " FO rate (dashed)")
        thr = [r.get("equivalent_score_threshold") if r["threshold_type"] != "score" else r["threshold"] for r in rr]
        ax[k][2].plot(RATIOS, [np.nan if t is None else t for t in thr], marker="o", color=col, label=rule)
    ax[k][0].set(xscale="log", xlabel="C_FV / C_FO", ylabel="test cost per 1000 decisions", title=v + ": cost-weighted error (TEST)")
    ax[k][1].set(xscale="log", xlabel="C_FV / C_FO", ylabel="rate", title=v + ": false-vacant / false-occupied rate (TEST)")
    ax[k][2].set(xscale="log", xlabel="C_FV / C_FO", ylabel="score threshold (or score equivalent)", title=v + ": decision threshold vs cost ratio")
    for a in ax[k]:
        a.grid(alpha=0.3); a.legend(fontsize=6)
plt.tight_layout(); plt.savefig(str(OUT / "cost_thresholds{}.png".format(SUF)), dpi=110); plt.show()

In [ ]:
# Record, environment, status
from src.common.env_record import save_environment
RECORD = {"run_id": RUN_ID, "config_hash": CFG_HASH, "smoke_test": SMOKE, "dataset_split_hash": SPLIT_INFO["manifest_hash"],
          "notebook07_run_id": st07["run_id"], "ratios_fv_over_fo": RATIOS, "cost_unit": "C_FO = 1; cost per 1000 slot decisions",
          "score_type": "uncalibrated detector score (max confidence of supporting detections)",
          "calibration": {v: {"chosen_by_val_log_loss": CALQ[v]["chosen"], "quality": {k: x for k, x in CALQ[v].items() if k != "chosen"},
                              "params": {m: CAL[v][m].params() for m in cs.CALIBRATORS}} for v in VARIANTS},
          "decision_rule_per_ratio": {v: {str(r): c for r, c in CHOICE[v].items()} for v in VARIANTS},
          "selection_rule": "per ratio, lowest {}-fold camera-day cross-fitted val cost among: nb07 score threshold, val score threshold, calibrated + Bayes".format(FOLDS),
          "test_rows": [r for r in ROWS if r["split"] == "test"],
          "temporal": {v: {str(r): {**c, "setting": tf.label(c["setting"])} for r, c in TCHOICE[v].items()} for v in VARIANTS},
          "temporal_test": {v: TEMP[v]["test"] for v in VARIANTS},
          "limitation": "cost ratios are illustrative (no measured cost of a wrong guidance); probabilities are calibrated on val (MetaPKLot sites); temporal part at the notebook 07 score thresholds"}
(OUT / "cost_sensitive_record{}.json".format(SUF)).write_text(json.dumps(RECORD, indent=2, default=str))
save_environment(str(OUT / "environment.json"))
write_status(STATUS, "11_cost_sensitive_occupancy_and_calibration", RUN_ID, "COMPLETE", CFG_HASH, "smoke={}".format(SMOKE),
             extra={v: {"calibrator": CALQ[v]["chosen"], "test_ece": CALQ[v][CALQ[v]["chosen"]]["test"]["ece"],
                        "rule_per_ratio": {str(r): c["rule"] for r, c in CHOICE[v].items()}} for v in VARIANTS})
print("COMPLETE | saved to", OUT, sorted(p.name for p in OUT.iterdir()))

## What to send back
Download the notebook after **Save & Run All** (SMOKE = False) and send it, or paste: the inputs cell, the calibration tables, the TEST cost table, the temporal cost table, and the plots.
`artifacts/cost_sensitive/` (calibrator parameters, rule per cost ratio) feeds notebooks 13-15, so **Save Version** with outputs.